# loading all the docs from corpus

In [5]:
import json

file_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\final_english_corpus.json"

with open(file_path, "r", encoding="utf-8") as f:
    english_corpus_data = json.load(f)

print("Total items:", len(english_corpus_data))


Total items: 426740


# counting decomposed claims 

In [6]:
import json

file_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-decomposed-claims.json"

with open(file_path, "r", encoding="utf-8") as f:
    decomposed_claims_data = json.load(f)

print(len(decomposed_claims_data))


2826


# main loop

# using BM25

In [7]:
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
import json
import re
import numpy as np
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm


# Run once if needed
# nltk.download("stopwords")
# nltk.download("wordnet")
# nltk.download("omw-1.4")

# ----------------------------
# Input / output paths
# ----------------------------

claims_output_path = r"E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-top100-BM25.json"

# ----------------------------
# removing stopwords and lemmatization setup
# ----------------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# ----------------------------
# Helper: simple tokenization
# ----------------------------

month_map = {
    "january": "01", "jan": "01",
    "february": "02", "feb": "02",
    "march": "03", "mar": "03",
    "april": "04", "apr": "04",
    "may": "05",
    "june": "06", "jun": "06",
    "july": "07", "jul": "07",
    "august": "08", "aug": "08",
    "september": "09", "sep": "09", "sept": "09",
    "october": "10", "oct": "10",
    "november": "11", "nov": "11",
    "december": "12", "dec": "12",
}

def normalize_numbers(text):
    text = str(text)

    # 10,000 -> 10000
    text = re.sub(r'(?<=\d),(?=\d)', '', text)

    # 10.5% -> 10.5 percent
    text = re.sub(r'(\d+(?:\.\d+)?)\s*%', r'\1 percent', text)

    # $1000 -> 1000 dollar
    text = re.sub(r'\$(\d+(?:\.\d+)?)', r'\1 dollar', text)

    # £250 -> 250 pound
    text = re.sub(r'£(\d+(?:\.\d+)?)', r'\1 pound', text)

    # €300 -> 300 euro
    text = re.sub(r'€(\d+(?:\.\d+)?)', r'\1 euro', text)

    return text

def normalize_dates(text):
    text = str(text)

    # April 10, 2025 -> 2025-04-10
    def replace_month_name_1(match):
        month = month_map[match.group(1).lower()]
        day = int(match.group(2))
        year = match.group(3)
        return f"{year}-{month}-{day:02d}"

    text = re.sub(
        r'\b(' + '|'.join(month_map.keys()) + r')\s+(\d{1,2}),\s*(\d{4})\b',
        replace_month_name_1,
        text,
        flags=re.IGNORECASE
    )

    # 10 April 2025 -> 2025-04-10
    def replace_month_name_2(match):
        day = int(match.group(1))
        month = month_map[match.group(2).lower()]
        year = match.group(3)
        return f"{year}-{month}-{day:02d}"

    text = re.sub(
        r'\b(\d{1,2})\s+(' + '|'.join(month_map.keys()) + r')\s+(\d{4})\b',
        replace_month_name_2,
        text,
        flags=re.IGNORECASE
    )

    # YYYY-MM-DD or YYYY/MM/DD -> YYYY-MM-DD
    text = re.sub(
        r'\b(\d{4})[-/](\d{1,2})[-/](\d{1,2})\b',
        lambda m: f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}",
        text
    )

    # DD/MM/YYYY or DD-MM-YYYY -> YYYY-MM-DD
    text = re.sub(
        r'\b(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b',
        lambda m: f"{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}",
        text
    )

    return text

def tokenize(text):
    text = str(text).lower()
    text = normalize_numbers(text)
    text = normalize_dates(text)

    # Keep decimals and yyyy-mm-dd as single tokens
    tokens = re.findall(r'\d{4}-\d{2}-\d{2}|\d+\.\d+|\w+', text)

    # Remove stopwords but keep numbers and normalized dates
    filtered_tokens = []
    for t in tokens:
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', t) or re.fullmatch(r'\d+(?:\.\d+)?', t):
            filtered_tokens.append(t)
        elif t not in stop_words:
            filtered_tokens.append(lemmatizer.lemmatize(t))

    return filtered_tokens

# ----------------------------
# Prepare corpus for BM25
# english_corpus_data is already loaded as:
# {doc_id: doc_text, ...}
# ----------------------------

doc_ids = list(english_corpus_data.keys())
doc_texts = [english_corpus_data[doc_id] for doc_id in doc_ids]
tokenized_corpus = [tokenize(text) for text in doc_texts]

bm25 = BM25Okapi(tokenized_corpus)

# ----------------------------
# Normalize decomposed_claims_data
# Supports either:
# 1. list of dicts
# 2. dict of dicts
# ----------------------------

if isinstance(decomposed_claims_data, dict):
    claim_items = []
    for k, v in decomposed_claims_data.items():
        if isinstance(v, dict):
            item = dict(v)
            item.setdefault("original_id", k)
            claim_items.append(item)
        else:
            claim_items.append({
                "original_id": k,
                "original_claim": str(v)
            })
elif isinstance(decomposed_claims_data, list):
    claim_items = decomposed_claims_data
else:
    raise TypeError("decomposed_claims_data must be a list or dict")

# ----------------------------
# BM25 retrieval
# ----------------------------

results = []

for item in tqdm(claim_items, desc="Processing claims"):
    original_claim = item.get("original_claim", "")
    original_id = item.get("original_id", item.get("id", None))

    query_tokens = tokenize(original_claim)
    scores = bm25.get_scores(query_tokens)

    top_k = min(100, len(doc_texts))
    top_indices = np.argsort(scores)[::-1][:top_k]

    top_docs = [doc_texts[i] for i in top_indices]
    top_docs_ids = [doc_ids[i] for i in top_indices]
    top_scores = [float(scores[i]) for i in top_indices]

    results.append({
        "original_id": original_id,
        "original_claim": original_claim,
        "doc_ids": top_docs_ids,
        "docs": top_docs,
        "scores": top_scores
    })
    

# ----------------------------
# Save output JSON
# ----------------------------

with open(claims_output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to {claims_output_path}")


Processing claims: 100%|██████████| 2826/2826 [1:14:59<00:00,  1.59s/it]


Saved 2826 results to E:\Projects\Numerical Fact Checking\CheckThat 2025\task3\data\1-my dataset\2-top100-BM25.json
